In [ ]:
# ============================================================
# CUADERNILLO DE CODIGO DEL PROYECTO CNN + XAI
# ============================================================

PROJECT_ROOT = '.'

# Estructura del cuadernillo:
# - EJERCICIOS 1 Y 2: Configuracion general del proyecto (src/settings.py)
# - EJERCICIOS 1 Y 2: Control de entorno y reproducibilidad (src/runtime.py)
# - EJERCICIO 1: Descarga y organizacion del dataset (scripts/download_dataset.py)
# - EJERCICIOS 1 Y 2: Exploracion, preprocesamiento y particiones (src/dataset.py)
# - EJERCICIO 3: Arquitectura CNN y utilidades de entrenamiento (src/modeling.py)
# - EJERCICIO 4: Pipeline completo de entrenamiento, ajuste y evaluacion (scripts/train_pipeline.py)
# - EJERCICIO 5: Generacion de saliency maps y Grad-CAM (src/xai.py)
# - EJERCICIO 5: Utilidades de visualizacion para resultados y figuras (src/visualization.py)
# - EJERCICIO 6: Preprocesamiento de inferencia para el despliegue (src/inference.py)
# - EJERCICIO 6: Aplicacion Streamlit desplegada (app/streamlit_app.py)

# Ejecucion manual sugerida:
# !python scripts/download_dataset.py
# !python scripts/train_pipeline.py
# !streamlit run app/streamlit_app.py

In [ ]:
# ============================================================
# EJERCICIOS 1 Y 2
# Configuracion general del proyecto
# Archivo: src/settings.py
# ============================================================

import os
from pathlib import Path

ROOT_DIR = Path(__file__).resolve().parent.parent
APP_DIR = ROOT_DIR / "app"
DATA_DIR = ROOT_DIR / "data"
MODELS_DIR = ROOT_DIR / "models"
NOTEBOOKS_DIR = ROOT_DIR / "notebooks"
REPORTS_DIR = ROOT_DIR / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
METRICS_DIR = REPORTS_DIR / "metrics"
SCRIPTS_DIR = ROOT_DIR / "scripts"
SRC_DIR = ROOT_DIR / "src"
LOCAL_CACHE_DIR = ROOT_DIR / ".cache"
EXPERIMENT_MODELS_DIR = MODELS_DIR / "experiments"

DEFAULT_IMAGE_SIZE = (96, 96)
BATCH_SIZE = 128
SEED = 42
CLASS_NAMES = ["female", "male"]
LABEL_TO_INDEX = {name: index for index, name in enumerate(CLASS_NAMES)}
INDEX_TO_LABEL = {index: name for name, index in LABEL_TO_INDEX.items()}

DATASET_HANDLE = "ashwingupta3012/male-and-female-faces-dataset"
MODELING_SAMPLES_PER_CLASS = int(os.environ.get("MODELING_SAMPLES_PER_CLASS", "900"))
KAGGLE_CACHE_ROOT = (
    Path.home()
    / ".cache"
    / "kagglehub"
    / "datasets"
    / "ashwingupta3012"
    / "male-and-female-faces-dataset"
    / "versions"
    / "1"
    / "Male and Female face dataset"
)

FINAL_MODEL_PATH = MODELS_DIR / "model.keras"
RESULTS_JSON_PATH = METRICS_DIR / "lab_results.json"


def ensure_project_dirs() -> None:
    for directory in [
        DATA_DIR,
        MODELS_DIR,
        NOTEBOOKS_DIR,
        REPORTS_DIR,
        FIGURES_DIR,
        METRICS_DIR,
        LOCAL_CACHE_DIR,
        EXPERIMENT_MODELS_DIR,
    ]:
        directory.mkdir(parents=True, exist_ok=True)


In [ ]:
# ============================================================
# EJERCICIOS 1 Y 2
# Control de entorno y reproducibilidad
# Archivo: src/runtime.py
# ============================================================

import os
import random

import numpy as np
import tensorflow as tf

from src.settings import LOCAL_CACHE_DIR, ensure_project_dirs


def configure_runtime() -> None:
    ensure_project_dirs()
    os.environ.setdefault("MPLCONFIGDIR", str(LOCAL_CACHE_DIR / "matplotlib"))
    os.environ.setdefault("XDG_CACHE_HOME", str(LOCAL_CACHE_DIR))
    os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
    (LOCAL_CACHE_DIR / "matplotlib").mkdir(parents=True, exist_ok=True)
    (LOCAL_CACHE_DIR / "fontconfig").mkdir(parents=True, exist_ok=True)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


In [ ]:
# ============================================================
# EJERCICIO 1
# Descarga y organizacion del dataset
# Archivo: scripts/download_dataset.py
# ============================================================

from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT_DIR = Path(__file__).resolve().parents[1]
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import kagglehub

from src.settings import DATASET_HANDLE, KAGGLE_CACHE_ROOT, DATA_DIR, ensure_project_dirs


def recreate_symlink(source: Path, target: Path) -> None:
    if target.exists() or target.is_symlink():
        if target.is_symlink() or target.is_file():
            target.unlink()
        else:
            raise RuntimeError(f"{target} ya existe y no es un enlace simbólico.")
    os.symlink(source, target)


def main() -> None:
    ensure_project_dirs()
    if not KAGGLE_CACHE_ROOT.exists():
        kagglehub.dataset_download(DATASET_HANDLE)

    source_male = KAGGLE_CACHE_ROOT / "Male Faces"
    source_female = KAGGLE_CACHE_ROOT / "Female Faces"

    if not source_male.exists() or not source_female.exists():
        raise FileNotFoundError(
            "La descarga se completó, pero no se encontraron las carpetas Male Faces/Female Faces."
        )

    recreate_symlink(source_male, DATA_DIR / "male")
    recreate_symlink(source_female, DATA_DIR / "female")

    print(f"Dataset listo en:\n- {DATA_DIR / 'male'}\n- {DATA_DIR / 'female'}")


if __name__ == "__main__":
    main()


In [ ]:
# ============================================================
# EJERCICIOS 1 Y 2
# Exploracion, preprocesamiento y particiones
# Archivo: src/dataset.py
# ============================================================

from __future__ import annotations

from collections import Counter
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image
from sklearn.model_selection import train_test_split

from src.settings import (
    CLASS_NAMES,
    DATA_DIR,
    DEFAULT_IMAGE_SIZE,
    LABEL_TO_INDEX,
    LOCAL_CACHE_DIR,
    SEED,
)

AUTOTUNE = tf.data.AUTOTUNE


def collect_image_records(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    records = []
    for label_name in CLASS_NAMES:
        class_dir = data_dir / label_name
        if not class_dir.exists():
            raise FileNotFoundError(
                f"No se encontró la carpeta {class_dir}. Ejecute scripts/download_dataset.py primero."
            )
        for path in sorted(class_dir.iterdir()):
            if path.is_file():
                records.append(
                    {
                        "filepath": str(path.resolve()),
                        "label": label_name,
                        "label_id": LABEL_TO_INDEX[label_name],
                        "extension": path.suffix.lower(),
                    }
                )
    if not records:
        raise RuntimeError("No se encontraron imágenes en data/male y data/female.")
    return pd.DataFrame(records)


def inspect_dataset(records: pd.DataFrame) -> Dict[str, Dict[str, object]]:
    summary: Dict[str, Dict[str, object]] = {}
    for label_name in CLASS_NAMES:
        class_df = records[records["label"] == label_name]
        widths = []
        heights = []
        modes: Counter[str] = Counter()
        extensions = Counter(class_df["extension"].tolist())
        bad_files = []
        for filepath in class_df["filepath"]:
            try:
                with Image.open(filepath) as image:
                    widths.append(image.width)
                    heights.append(image.height)
                    modes[image.mode] += 1
            except Exception:
                bad_files.append(Path(filepath).name)

        summary[label_name] = {
            "count": int(len(class_df)),
            "extensions": dict(extensions),
            "modes": dict(modes),
            "bad_files": bad_files,
            "width_min": int(min(widths)),
            "width_max": int(max(widths)),
            "width_mean": float(np.mean(widths)),
            "width_median": float(np.median(widths)),
            "height_min": int(min(heights)),
            "height_max": int(max(heights)),
            "height_mean": float(np.mean(heights)),
            "height_median": float(np.median(heights)),
        }
    return summary


def make_stratified_splits(
    records: pd.DataFrame,
    seed: int = SEED,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_df, temp_df = train_test_split(
        records,
        test_size=0.30,
        random_state=seed,
        stratify=records["label_id"],
    )
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=seed,
        stratify=temp_df["label_id"],
    )
    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )


def _decode_and_resize(path: tf.Tensor, label: tf.Tensor, image_size: Tuple[int, int]):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.convert_image_dtype(image, tf.float32)
    image = tf.image.resize_with_pad(image, image_size[0], image_size[1])
    return image, tf.cast(label, tf.float32)


def build_tf_dataset(
    records: pd.DataFrame,
    image_size: Tuple[int, int] = DEFAULT_IMAGE_SIZE,
    batch_size: int = 32,
    training: bool = False,
) -> tf.data.Dataset:
    paths = records["filepath"].astype(str).to_numpy()
    labels = records["label_id"].astype("float32").to_numpy()

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(len(records), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(
        lambda path, label: _decode_and_resize(path, label, image_size),
        num_parallel_calls=AUTOTUNE,
    )
    dataset = dataset.batch(batch_size).prefetch(AUTOTUNE)
    return dataset


def _preprocess_rgb_array(rgb_array: np.ndarray, image_size: Tuple[int, int]) -> np.ndarray:
    tensor = tf.convert_to_tensor(rgb_array, dtype=tf.uint8)
    tensor = tf.cast(tensor, tf.float32)
    tensor = tf.image.resize_with_pad(tensor, image_size[0], image_size[1])
    tensor = tf.cast(tf.clip_by_value(tf.round(tensor), 0, 255), tf.uint8)
    return tensor.numpy()


def load_or_create_array_cache(
    records: pd.DataFrame,
    split_name: str,
    image_size: Tuple[int, int] = DEFAULT_IMAGE_SIZE,
):
    cache_dir = LOCAL_CACHE_DIR / "preprocessed"
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_file = cache_dir / f"{split_name}_{image_size[0]}x{image_size[1]}.npz"

    if cache_file.exists():
        with np.load(cache_file) as cached:
            images = cached["images"]
            labels = cached["labels"]
            if len(labels) == len(records):
                return images, labels

    images = np.empty((len(records), image_size[0], image_size[1], 3), dtype=np.uint8)
    labels = records["label_id"].astype("float32").to_numpy()
    for index, filepath in enumerate(records["filepath"]):
        if index % 500 == 0:
            print(f"[{split_name}] procesadas {index}/{len(records)} imágenes", flush=True)
        with Image.open(filepath) as image:
            rgb_array = np.array(image.convert("RGB"))
        images[index] = _preprocess_rgb_array(rgb_array, image_size)

    np.savez(cache_file, images=images, labels=labels)
    return images, labels


def build_array_dataset(
    images: np.ndarray,
    labels: np.ndarray,
    batch_size: int,
    training: bool = False,
) -> tf.data.Dataset:
    dataset = tf.data.Dataset.from_tensor_slices((images, labels))
    if training:
        dataset = dataset.shuffle(len(labels), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(
        lambda image, label: (tf.cast(image, tf.float32) / 255.0, tf.cast(label, tf.float32)),
        num_parallel_calls=AUTOTUNE,
    )
    dataset = dataset.batch(batch_size).prefetch(AUTOTUNE)
    return dataset


def prepare_image_from_pil(
    pil_image: Image.Image,
    image_size: Tuple[int, int] = DEFAULT_IMAGE_SIZE,
):
    rgb_image = pil_image.convert("RGB")
    rgb_array = np.array(rgb_image)
    tensor = tf.convert_to_tensor(rgb_array, dtype=tf.uint8)
    tensor = tf.image.convert_image_dtype(tensor, tf.float32)
    tensor = tf.image.resize_with_pad(tensor, image_size[0], image_size[1])
    tensor = tf.expand_dims(tensor, axis=0)
    resized_rgb = tf.image.convert_image_dtype(tensor[0], tf.uint8).numpy()
    return rgb_array, resized_rgb, tensor


In [ ]:
# ============================================================
# EJERCICIO 3
# Arquitectura CNN y utilidades de entrenamiento
# Archivo: src/modeling.py
# ============================================================

from __future__ import annotations

from io import StringIO
from pathlib import Path
from typing import Dict, Iterable

import pandas as pd
import tensorflow as tf

from src.settings import DEFAULT_IMAGE_SIZE


def build_cnn(
    input_shape=(DEFAULT_IMAGE_SIZE[0], DEFAULT_IMAGE_SIZE[1], 3),
    filters=(8, 16, 32),
    kernel_size=3,
    dense_units=64,
    dropout_rate=0.30,
    learning_rate=1e-3,
) -> tf.keras.Model:
    inputs = tf.keras.Input(shape=input_shape, name="input_image")
    x = inputs

    for index, n_filters in enumerate(filters, start=1):
        x = tf.keras.layers.Conv2D(
            n_filters,
            kernel_size,
            padding="same",
            activation="relu",
            name=f"conv_block_{index}_conv",
        )(x)
        x = tf.keras.layers.MaxPooling2D(pool_size=2, name=f"conv_block_{index}_pool")(x)

    x = tf.keras.layers.Conv2D(
        filters[-1],
        3,
        padding="same",
        activation="relu",
        name="last_conv",
    )(x)
    x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(x)
    x = tf.keras.layers.Dense(dense_units, activation="relu", name="dense_1")(x)
    x = tf.keras.layers.Dropout(dropout_rate, name="dropout")(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid", name="prediction")(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="cnn_gender_classifier")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.AUC(name="auc"),
        ],
        run_eagerly=True,
    )
    return model


def experiment_configs() -> Iterable[Dict[str, object]]:
    return [
        {
            "name": "baseline",
            "filters": (8, 16, 32),
            "kernel_size": 3,
            "dense_units": 64,
            "dropout_rate": 0.20,
            "learning_rate": 1e-3,
            "epochs": 6,
        },
        {
            "name": "regularized",
            "filters": (8, 16, 32),
            "kernel_size": 3,
            "dense_units": 64,
            "dropout_rate": 0.35,
            "learning_rate": 5e-4,
            "epochs": 8,
        },
    ]


def build_callbacks(checkpoint_path: Path):
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            verbose=1,
            min_lr=1e-6,
        ),
        tf.keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_path,
            monitor="val_loss",
            save_best_only=True,
            verbose=1,
        ),
    ]


def capture_model_summary(model: tf.keras.Model) -> str:
    buffer = StringIO()
    model.summary(print_fn=lambda line: buffer.write(f"{line}\n"))
    return buffer.getvalue()


def history_to_frame(history: tf.keras.callbacks.History) -> pd.DataFrame:
    frame = pd.DataFrame(history.history)
    frame.index = frame.index + 1
    frame.index.name = "epoch"
    return frame.reset_index()


In [ ]:
# ============================================================
# EJERCICIO 4
# Pipeline completo de entrenamiento, ajuste y evaluacion
# Archivo: scripts/train_pipeline.py
# ============================================================

from __future__ import annotations

import json
import sys
from pathlib import Path

ROOT_DIR = Path(__file__).resolve().parents[1]
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report

from src.dataset import (
    collect_image_records,
    inspect_dataset,
    load_or_create_array_cache,
    make_stratified_splits,
)
from src.inference import prepare_image_from_pil
from src.modeling import (
    build_callbacks,
    build_cnn,
    capture_model_summary,
    experiment_configs,
    history_to_frame,
)
from src.runtime import configure_runtime, seed_everything
from src.settings import (
    BATCH_SIZE,
    CLASS_NAMES,
    DEFAULT_IMAGE_SIZE,
    EXPERIMENT_MODELS_DIR,
    FIGURES_DIR,
    FINAL_MODEL_PATH,
    METRICS_DIR,
    MODELING_SAMPLES_PER_CLASS,
    RESULTS_JSON_PATH,
    SEED,
    ensure_project_dirs,
)
from src.visualization import (
    save_class_distribution,
    save_confusion_matrix,
    save_dataset_mosaic,
    save_hyperparameter_comparison,
    save_training_curves,
    save_xai_figure,
)
from src.xai import make_gradcam_heatmap, make_saliency_map, overlay_heatmap


def serialize_inspection(inspection: dict) -> dict:
    serializable = {}
    for label_name, values in inspection.items():
        serializable[label_name] = {
            key: (value if not isinstance(value, np.generic) else value.item())
            for key, value in values.items()
        }
    return serializable


def train_experiment(
    config: dict,
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_val: np.ndarray,
    y_val: np.ndarray,
):
    tf.keras.backend.clear_session()
    model = build_cnn(
        filters=config["filters"],
        kernel_size=config["kernel_size"],
        dense_units=config["dense_units"],
        dropout_rate=config["dropout_rate"],
        learning_rate=config["learning_rate"],
    )
    checkpoint_path = EXPERIMENT_MODELS_DIR / f"{config['name']}.keras"
    history = model.fit(
        x_train,
        y_train,
        validation_data=(x_val, y_val),
        batch_size=BATCH_SIZE,
        epochs=config["epochs"],
        callbacks=build_callbacks(checkpoint_path),
        verbose=2,
    )
    best_model = tf.keras.models.load_model(checkpoint_path)
    history_df = history_to_frame(history)
    best_row = history_df.loc[history_df["val_accuracy"].idxmax()]
    summary = {
        **config,
        "best_epoch": int(best_row["epoch"]),
        "best_val_accuracy": float(best_row["val_accuracy"]),
        "best_val_loss": float(best_row["val_loss"]),
        "final_train_accuracy": float(history_df.iloc[-1]["accuracy"]),
        "final_train_loss": float(history_df.iloc[-1]["loss"]),
    }
    return best_model, history_df, summary


def choose_correct_sample(model: tf.keras.Model, test_df: pd.DataFrame):
    candidates = []
    for _, row in test_df.iterrows():
        with Path(row["filepath"]).open("rb") as stream:
            from PIL import Image

            image = Image.open(stream)
            original_rgb, resized_rgb, input_tensor = prepare_image_from_pil(image, DEFAULT_IMAGE_SIZE)
        prob_male = float(model.predict(input_tensor, verbose=0)[0][0])
        predicted_id = int(prob_male >= 0.5)
        confidence = float(max(prob_male, 1 - prob_male))
        if predicted_id == int(row["label_id"]):
            candidates.append(
                {
                    "filepath": row["filepath"],
                    "label": row["label"],
                    "label_id": int(row["label_id"]),
                    "predicted_id": predicted_id,
                    "prob_male": prob_male,
                    "confidence": confidence,
                    "original_rgb": original_rgb,
                    "resized_rgb": resized_rgb,
                    "input_tensor": input_tensor,
                }
            )
    if not candidates:
        raise RuntimeError("No se encontró una imagen correctamente clasificada para generar XAI.")
    return max(candidates, key=lambda item: item["confidence"])


def run_pipeline() -> dict:
    configure_runtime()
    ensure_project_dirs()
    seed_everything(SEED)

    records = collect_image_records()
    inspection = inspect_dataset(records)
    modeling_records = (
        pd.concat(
            [
                group.sample(n=min(len(group), MODELING_SAMPLES_PER_CLASS), random_state=SEED)
                for _, group in records.groupby("label_id")
            ]
        )
        .sample(frac=1.0, random_state=SEED)
        .reset_index(drop=True)
    )
    train_df, val_df, test_df = make_stratified_splits(modeling_records)
    print(f"Total de imágenes: {len(records)}", flush=True)
    print(
        f"Subset estratificado para modelado: {len(modeling_records)} "
        f"({MODELING_SAMPLES_PER_CLASS} por clase como máximo)",
        flush=True,
    )
    print(
        f"Splits -> train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}",
        flush=True,
    )

    save_dataset_mosaic(records, FIGURES_DIR / "dataset_mosaic.png")
    save_class_distribution(records, FIGURES_DIR / "class_distribution.png")

    print("Preprocesando/cargando caché del split de entrenamiento...", flush=True)
    train_images, train_labels = load_or_create_array_cache(train_df, "train", DEFAULT_IMAGE_SIZE)
    print("Preprocesando/cargando caché del split de validación...", flush=True)
    val_images, val_labels = load_or_create_array_cache(val_df, "validation", DEFAULT_IMAGE_SIZE)
    print("Preprocesando/cargando caché del split de prueba...", flush=True)
    test_images, test_labels = load_or_create_array_cache(test_df, "test", DEFAULT_IMAGE_SIZE)
    print("Convirtiendo arreglos a float32 normalizado...", flush=True)
    train_images = train_images.astype("float32") / 255.0
    val_images = val_images.astype("float32") / 255.0
    test_images = test_images.astype("float32") / 255.0

    experiment_indices = (
        train_df.groupby("label_id", group_keys=False)
        .sample(frac=0.50, random_state=SEED)
        .index.to_numpy()
    )
    experiment_train_images = train_images[experiment_indices]
    experiment_train_labels = train_labels[experiment_indices]

    experiment_rows = []
    for config in experiment_configs():
        print(f"Iniciando experimento: {config['name']}", flush=True)
        model, history_df, summary = train_experiment(
            config,
            experiment_train_images,
            experiment_train_labels,
            val_images,
            val_labels,
        )
        save_training_curves(
            history_df,
            FIGURES_DIR / f"training_{config['name']}.png",
            title=f"Curvas de entrenamiento - {config['name']}",
        )
        experiment_rows.append(summary)

    experiments_df = pd.DataFrame(experiment_rows).sort_values(
        by=["best_val_accuracy", "best_val_loss"],
        ascending=[False, True],
    )
    save_hyperparameter_comparison(experiments_df, FIGURES_DIR / "hyperparameter_comparison.png")
    experiments_df.to_csv(METRICS_DIR / "hyperparameter_results.csv", index=False)

    best_name = experiments_df.iloc[0]["name"]
    best_config = next(config for config in experiment_configs() if config["name"] == best_name)
    print(f"Mejor configuración preliminar: {best_name}", flush=True)
    tf.keras.backend.clear_session()
    final_model = build_cnn(
        filters=best_config["filters"],
        kernel_size=best_config["kernel_size"],
        dense_units=best_config["dense_units"],
        dropout_rate=best_config["dropout_rate"],
        learning_rate=best_config["learning_rate"],
    )
    final_history = final_model.fit(
        train_images,
        train_labels,
        validation_data=(val_images, val_labels),
        batch_size=BATCH_SIZE,
        epochs=max(8, best_config["epochs"]),
        callbacks=build_callbacks(FINAL_MODEL_PATH),
        verbose=2,
    )
    print("Evaluando modelo final y generando artefactos XAI...", flush=True)
    final_model = tf.keras.models.load_model(FINAL_MODEL_PATH)
    final_history_df = history_to_frame(final_history)
    save_training_curves(
        final_history_df,
        FIGURES_DIR / "training_final.png",
        title="Curvas de entrenamiento - modelo final",
    )
    final_model.save(FINAL_MODEL_PATH)
    (METRICS_DIR / "model_summary.txt").write_text(capture_model_summary(final_model), encoding="utf-8")

    test_metrics = final_model.evaluate(test_images, test_labels, batch_size=BATCH_SIZE, return_dict=True, verbose=0)
    probabilities = final_model.predict(test_images, batch_size=BATCH_SIZE, verbose=0).ravel()
    predictions = (probabilities >= 0.5).astype(int)
    y_true = test_labels.astype(int)
    save_confusion_matrix(y_true, predictions, FIGURES_DIR / "confusion_matrix.png")

    report = classification_report(
        y_true,
        predictions,
        target_names=CLASS_NAMES,
        output_dict=True,
    )
    (METRICS_DIR / "classification_report.json").write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    chosen = choose_correct_sample(final_model, test_df)
    saliency = make_saliency_map(final_model, chosen["input_tensor"])
    gradcam = make_gradcam_heatmap(final_model, chosen["input_tensor"], last_conv_layer_name="last_conv")
    saliency_overlay = overlay_heatmap(chosen["resized_rgb"], saliency)
    gradcam_overlay = overlay_heatmap(chosen["resized_rgb"], gradcam)
    save_xai_figure(
        chosen["resized_rgb"],
        saliency_overlay,
        gradcam_overlay,
        FIGURES_DIR / "xai_example.png",
        title="Interpretabilidad visual sobre una imagen correctamente clasificada",
    )

    results = {
        "dataset_summary": serialize_inspection(inspection),
        "split_counts": {
            "train": int(len(train_df)),
            "validation": int(len(val_df)),
            "test": int(len(test_df)),
        },
        "modeling_dataset_count": int(len(modeling_records)),
        "modeling_samples_per_class": int(MODELING_SAMPLES_PER_CLASS),
        "image_size": list(DEFAULT_IMAGE_SIZE),
        "batch_size": BATCH_SIZE,
        "experiments": experiments_df.to_dict(orient="records"),
        "best_experiment": experiments_df.iloc[0].to_dict(),
        "test_metrics": {key: float(value) for key, value in test_metrics.items()},
        "classification_report": report,
        "xai_sample": {
            "filepath": chosen["filepath"],
            "true_label": chosen["label"],
            "predicted_label": CLASS_NAMES[chosen["predicted_id"]],
            "prob_male": float(chosen["prob_male"]),
            "confidence": float(chosen["confidence"]),
        },
        "artifacts": {
            "mosaic": str(FIGURES_DIR / "dataset_mosaic.png"),
            "class_distribution": str(FIGURES_DIR / "class_distribution.png"),
            "hyperparameter_comparison": str(FIGURES_DIR / "hyperparameter_comparison.png"),
            "confusion_matrix": str(FIGURES_DIR / "confusion_matrix.png"),
            "xai_example": str(FIGURES_DIR / "xai_example.png"),
        },
    }

    RESULTS_JSON_PATH.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
    return results


if __name__ == "__main__":
    summary = run_pipeline()
    print(json.dumps(summary["test_metrics"], indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# EJERCICIO 5
# Generacion de saliency maps y Grad-CAM
# Archivo: src/xai.py
# ============================================================

from __future__ import annotations

import numpy as np
import tensorflow as tf
from matplotlib import cm


def make_saliency_map(model: tf.keras.Model, image_tensor: tf.Tensor) -> np.ndarray:
    image_tensor = tf.cast(image_tensor, tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(image_tensor)
        predictions = model(image_tensor, training=False)
        target = predictions[:, 0]
    gradients = tape.gradient(target, image_tensor)[0]
    saliency = tf.reduce_max(tf.abs(gradients), axis=-1)
    saliency = saliency.numpy()
    saliency -= saliency.min()
    if saliency.max() > 0:
        saliency /= saliency.max()
    return saliency


def make_gradcam_heatmap(
    model: tf.keras.Model,
    image_tensor: tf.Tensor,
    last_conv_layer_name: str = "last_conv",
) -> np.ndarray:
    grad_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output],
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(image_tensor, training=False)
        target = predictions[:, 0]

    gradients = tape.gradient(target, conv_outputs)
    pooled_gradients = tf.reduce_mean(gradients, axis=(1, 2))
    conv_outputs = conv_outputs[0]
    pooled_gradients = pooled_gradients[0]

    heatmap = conv_outputs @ pooled_gradients[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    heatmap = heatmap.numpy()
    heatmap = np.nan_to_num(heatmap)
    return heatmap


def overlay_heatmap(
    image_rgb: np.ndarray,
    heatmap: np.ndarray,
    alpha: float = 0.40,
) -> np.ndarray:
    heatmap = tf.image.resize(
        heatmap[..., np.newaxis],
        (image_rgb.shape[0], image_rgb.shape[1]),
        method="bilinear",
    ).numpy().squeeze()
    heatmap = np.clip(heatmap, 0.0, 1.0)
    colored = (cm.get_cmap("jet")(heatmap)[..., :3] * 255).astype(np.uint8)
    overlay = ((1 - alpha) * image_rgb.astype(np.float32) + alpha * colored.astype(np.float32)).astype(
        np.uint8
    )
    return overlay


In [ ]:
# ============================================================
# EJERCICIO 5
# Utilidades de visualizacion para resultados y figuras
# Archivo: src/visualization.py
# ============================================================

from __future__ import annotations

from pathlib import Path
from typing import Dict

from src.runtime import configure_runtime

configure_runtime()

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

from src.settings import CLASS_NAMES


def save_dataset_mosaic(records: pd.DataFrame, output_path: Path, samples_per_class: int = 6) -> None:
    figure, axes = plt.subplots(len(CLASS_NAMES), samples_per_class, figsize=(14, 5))
    for row, label_name in enumerate(CLASS_NAMES):
        sample_df = records[records["label"] == label_name].head(samples_per_class)
        for col, (_, item) in enumerate(sample_df.iterrows()):
            with Image.open(item["filepath"]) as image:
                axes[row, col].imshow(image.convert("RGB"))
            axes[row, col].set_title(label_name.capitalize())
            axes[row, col].axis("off")
    figure.suptitle("Mosaico representativo del dataset", fontsize=16, weight="bold")
    figure.tight_layout()
    figure.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(figure)


def save_class_distribution(records: pd.DataFrame, output_path: Path) -> None:
    counts = records["label"].value_counts().reindex(CLASS_NAMES)
    figure, ax = plt.subplots(figsize=(6, 4))
    colors = ["#ff8fab", "#4c78a8"]
    bars = ax.bar(counts.index, counts.values, color=colors)
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 20,
            f"{int(bar.get_height())}",
            ha="center",
            va="bottom",
            fontsize=11,
        )
    ax.set_title("Distribución por clase")
    ax.set_ylabel("Número de imágenes")
    figure.tight_layout()
    figure.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(figure)


def save_training_curves(history_df: pd.DataFrame, output_path: Path, title: str) -> None:
    figure, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history_df["epoch"], history_df["loss"], label="Entrenamiento")
    axes[0].plot(history_df["epoch"], history_df["val_loss"], label="Validación")
    axes[0].set_title("Pérdida")
    axes[0].set_xlabel("Época")
    axes[0].set_ylabel("Binary crossentropy")
    axes[0].legend()

    axes[1].plot(history_df["epoch"], history_df["accuracy"], label="Entrenamiento")
    axes[1].plot(history_df["epoch"], history_df["val_accuracy"], label="Validación")
    axes[1].set_title("Exactitud")
    axes[1].set_xlabel("Época")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()

    figure.suptitle(title, fontsize=15, weight="bold")
    figure.tight_layout()
    figure.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(figure)


def save_confusion_matrix(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    output_path: Path,
) -> None:
    matrix = confusion_matrix(y_true, y_pred)
    figure, ax = plt.subplots(figsize=(5, 5))
    ConfusionMatrixDisplay(matrix, display_labels=CLASS_NAMES).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title("Matriz de confusión en prueba")
    figure.tight_layout()
    figure.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(figure)


def save_hyperparameter_comparison(results_df: pd.DataFrame, output_path: Path) -> None:
    figure, ax = plt.subplots(figsize=(7, 4))
    ax.bar(results_df["name"], results_df["best_val_accuracy"], color=["#4c78a8", "#f58518"])
    for _, row in results_df.iterrows():
        ax.text(row["name"], row["best_val_accuracy"] + 0.002, f"{row['best_val_accuracy']:.3f}", ha="center")
    ax.set_ylim(0.0, min(1.0, results_df["best_val_accuracy"].max() + 0.08))
    ax.set_title("Comparación de configuraciones")
    ax.set_ylabel("Mejor accuracy de validación")
    figure.tight_layout()
    figure.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(figure)


def save_xai_figure(
    original_rgb: np.ndarray,
    saliency_overlay: np.ndarray,
    gradcam_overlay: np.ndarray,
    output_path: Path,
    title: str,
) -> None:
    figure, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    axes[0].imshow(original_rgb)
    axes[0].set_title("Imagen preprocesada")
    axes[1].imshow(saliency_overlay)
    axes[1].set_title("Saliency Map")
    axes[2].imshow(gradcam_overlay)
    axes[2].set_title("Grad-CAM")
    for ax in axes:
        ax.axis("off")
    figure.suptitle(title, fontsize=15, weight="bold")
    figure.tight_layout()
    figure.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(figure)


def save_markdown_table(results_df: pd.DataFrame, output_path: Path) -> None:
    headers = list(results_df.columns)
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    for _, row in results_df.iterrows():
        formatted = [str(value) for value in row.tolist()]
        lines.append("| " + " | ".join(formatted) + " |")
    output_path.write_text("\n".join(lines), encoding="utf-8")


In [ ]:
# ============================================================
# EJERCICIO 6
# Preprocesamiento de inferencia para el despliegue
# Archivo: src/inference.py
# ============================================================

from __future__ import annotations

from typing import Tuple

import numpy as np
import tensorflow as tf
from PIL import Image

from src.settings import DEFAULT_IMAGE_SIZE


def prepare_image_from_pil(
    pil_image: Image.Image,
    image_size: Tuple[int, int] = DEFAULT_IMAGE_SIZE,
):
    rgb_image = pil_image.convert("RGB")
    rgb_array = np.array(rgb_image)
    tensor = tf.convert_to_tensor(rgb_array, dtype=tf.uint8)
    tensor = tf.image.convert_image_dtype(tensor, tf.float32)
    tensor = tf.image.resize_with_pad(tensor, image_size[0], image_size[1])
    tensor = tf.expand_dims(tensor, axis=0)
    resized_rgb = tf.image.convert_image_dtype(tensor[0], tf.uint8).numpy()
    return rgb_array, resized_rgb, tensor


In [ ]:
# ============================================================
# EJERCICIO 6
# Aplicacion Streamlit desplegada
# Archivo: app/streamlit_app.py
# ============================================================

from __future__ import annotations

import base64
import html
import io
import json
import re
import sys
from pathlib import Path
from typing import Any

import numpy as np

ROOT_DIR = Path(__file__).resolve().parents[1]
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import streamlit as st
import tensorflow as tf
from PIL import Image

from src.inference import prepare_image_from_pil
from src.settings import CLASS_NAMES, DEFAULT_IMAGE_SIZE, FINAL_MODEL_PATH, RESULTS_JSON_PATH
from src.xai import make_gradcam_heatmap, make_saliency_map, overlay_heatmap

MODEL_SUMMARY_PATH = ROOT_DIR / "reports" / "metrics" / "model_summary.txt"
FIGURE_PATHS = {
    "class_distribution": ROOT_DIR / "reports" / "figures" / "class_distribution.png",
    "confusion_matrix": ROOT_DIR / "reports" / "figures" / "confusion_matrix.png",
    "dataset_mosaic": ROOT_DIR / "reports" / "figures" / "dataset_mosaic.png",
    "hyperparameter_comparison": ROOT_DIR / "reports" / "figures" / "hyperparameter_comparison.png",
    "training_final": ROOT_DIR / "reports" / "figures" / "training_final.png",
    "training_regularized": ROOT_DIR / "reports" / "figures" / "training_regularized.png",
    "xai_example": ROOT_DIR / "reports" / "figures" / "xai_example.png",
}

st.set_page_config(
    page_title="Prisma XAI Lab | Clasificacion por sexo con CNN",
    page_icon="🔬",
    layout="wide",
    initial_sidebar_state="collapsed",
)

CUSTOM_CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@400;500;700&family=Hanken+Grotesk:wght@400;500;600;700&display=swap');

:root {
    --bg-0: #040a11;
    --bg-1: #08111c;
    --bg-2: #0c1723;
    --surface: rgba(10, 18, 27, 0.92);
    --surface-soft: rgba(14, 23, 34, 0.84);
    --stroke: rgba(136, 221, 233, 0.16);
    --stroke-strong: rgba(109, 247, 255, 0.42);
    --cyan: #4defff;
    --cyan-bright: #7ef7ff;
    --text: #e7fbff;
    --muted: #88aab6;
    --teal: #11343b;
    --shadow: 0 30px 80px rgba(0, 0, 0, 0.42);
}

html, body, [class*="css"] {
    font-family: "Hanken Grotesk", sans-serif;
}

body {
    color: var(--text);
}

#MainMenu,
footer,
header[data-testid="stHeader"],
[data-testid="collapsedControl"],
[data-testid="stToolbar"],
[data-testid="stSidebar"] {
    display: none !important;
}

[data-testid="stAppViewContainer"] {
    background:
        radial-gradient(circle at 15% 0%, rgba(58, 163, 184, 0.18), transparent 28%),
        radial-gradient(circle at 85% 12%, rgba(65, 196, 219, 0.12), transparent 24%),
        linear-gradient(180deg, #03070d 0%, #07111a 34%, #050b12 100%);
    color: var(--text);
}

[data-testid="stAppViewContainer"]::before {
    content: "";
    position: fixed;
    inset: 0;
    pointer-events: none;
    background-image:
        linear-gradient(rgba(126, 247, 255, 0.045) 1px, transparent 1px),
        linear-gradient(90deg, rgba(126, 247, 255, 0.045) 1px, transparent 1px);
    background-size: 48px 48px;
    mask-image: linear-gradient(180deg, transparent 0%, black 20%, black 88%, transparent 100%);
    opacity: 0.18;
}

main .block-container {
    max-width: 1220px;
    padding-top: 0.7rem;
    padding-bottom: 4rem;
    position: relative;
    z-index: 1;
}

a {
    color: inherit;
}

.section-anchor {
    position: relative;
    top: -96px;
    visibility: hidden;
}

.top-nav {
    position: sticky;
    top: 0.9rem;
    z-index: 12;
    display: flex;
    align-items: center;
    justify-content: space-between;
    gap: 1rem;
    padding: 1rem 1.2rem;
    margin-bottom: 1rem;
    border-radius: 22px;
    background: linear-gradient(180deg, rgba(10, 18, 28, 0.92), rgba(7, 12, 20, 0.92));
    border: 1px solid rgba(136, 221, 233, 0.12);
    box-shadow: 0 16px 50px rgba(0, 0, 0, 0.26);
    backdrop-filter: blur(18px);
}

.brand-lockup {
    font-family: "Space Grotesk", sans-serif;
    font-size: 1.7rem;
    font-weight: 700;
    letter-spacing: -0.04em;
    color: #dffaff;
    text-decoration: none;
    white-space: nowrap;
}

.nav-links {
    display: flex;
    align-items: center;
    justify-content: center;
    gap: 1.25rem;
    flex: 1;
    flex-wrap: wrap;
}

.nav-links a {
    text-decoration: none;
    text-transform: uppercase;
    letter-spacing: 0.16em;
    font-size: 0.72rem;
    color: var(--muted);
    transition: color 0.2s ease, transform 0.2s ease;
}

.nav-links a:hover {
    color: var(--text);
    transform: translateY(-1px);
}

.nav-cta {
    text-decoration: none;
    padding: 0.8rem 1rem;
    border-radius: 14px;
    border: 1px solid rgba(77, 239, 255, 0.55);
    background: linear-gradient(135deg, #63f3ff 0%, #15d8f6 100%);
    color: #041017;
    font-weight: 700;
    font-size: 0.75rem;
    letter-spacing: 0.14em;
    text-transform: uppercase;
    box-shadow: 0 0 0 1px rgba(109, 247, 255, 0.08), 0 10px 28px rgba(24, 214, 243, 0.28);
}

.hero-shell {
    position: relative;
    overflow: hidden;
    padding: clamp(2.6rem, 5vw, 4.2rem);
    min-height: 31rem;
    border-radius: 34px;
    background:
        linear-gradient(130deg, rgba(5, 17, 24, 0.95) 0%, rgba(6, 20, 29, 0.88) 38%, rgba(8, 24, 31, 0.82) 100%),
        radial-gradient(circle at 50% 0%, rgba(75, 223, 255, 0.12), transparent 34%);
    border: 1px solid rgba(136, 221, 233, 0.14);
    box-shadow: var(--shadow);
    margin-bottom: 3rem;
}

.hero-shell::before {
    content: "";
    position: absolute;
    inset: -10% -10% 14% -10%;
    background:
        linear-gradient(115deg, transparent 0%, rgba(79, 237, 255, 0.12) 18%, transparent 32%),
        linear-gradient(125deg, transparent 10%, rgba(79, 237, 255, 0.06) 24%, transparent 42%),
        repeating-linear-gradient(110deg, rgba(86, 241, 255, 0.05) 0 3px, transparent 3px 18px);
    animation: heroDrift 18s linear infinite;
    opacity: 0.65;
}

.hero-shell::after {
    content: "";
    position: absolute;
    inset: auto -15% -30% 25%;
    height: 24rem;
    background: radial-gradient(circle, rgba(77, 239, 255, 0.18), transparent 58%);
    filter: blur(18px);
    pointer-events: none;
}

.hero-content {
    position: relative;
    z-index: 1;
    max-width: 760px;
}

.hero-badge {
    display: inline-flex;
    align-items: center;
    gap: 0.5rem;
    padding: 0.45rem 0.9rem;
    margin-bottom: 1rem;
    border-radius: 999px;
    border: 1px solid rgba(109, 247, 255, 0.18);
    background: rgba(5, 20, 27, 0.72);
    color: var(--cyan-bright);
    font-size: 0.74rem;
    letter-spacing: 0.18em;
    text-transform: uppercase;
}

.hero-title {
    margin: 0;
    font-family: "Space Grotesk", sans-serif;
    font-size: clamp(3.2rem, 8vw, 5.9rem);
    line-height: 0.94;
    letter-spacing: -0.06em;
    color: #e9fdff;
    text-shadow: 0 0 30px rgba(79, 237, 255, 0.08), 0 18px 40px rgba(0, 0, 0, 0.28);
}

.hero-title span {
    color: var(--cyan-bright);
}

.hero-copy {
    margin: 1.25rem 0 0;
    max-width: 620px;
    color: #b4d4db;
    font-size: 1.04rem;
    line-height: 1.75;
}

.hero-actions {
    display: flex;
    flex-wrap: wrap;
    gap: 0.85rem;
    margin-top: 2rem;
}

.hero-button {
    text-decoration: none;
    display: inline-flex;
    align-items: center;
    justify-content: center;
    gap: 0.45rem;
    min-width: 220px;
    padding: 0.95rem 1.2rem;
    border-radius: 16px;
    text-transform: uppercase;
    letter-spacing: 0.16em;
    font-weight: 700;
    font-size: 0.76rem;
}

.hero-button.primary {
    background: linear-gradient(135deg, #63f3ff 0%, #12d7f4 100%);
    color: #031017;
    border: 1px solid rgba(77, 239, 255, 0.55);
    box-shadow: 0 0 0 1px rgba(109, 247, 255, 0.08), 0 14px 34px rgba(24, 214, 243, 0.24);
}

.hero-button.secondary {
    background: rgba(9, 18, 28, 0.72);
    color: #d6fbff;
    border: 1px solid rgba(136, 221, 233, 0.18);
}

.hero-meta {
    position: relative;
    z-index: 1;
    display: grid;
    grid-template-columns: repeat(4, minmax(0, 1fr));
    gap: 0.9rem;
    margin-top: 2.4rem;
}

.hero-meta-card,
.metric-card,
.glass-card {
    position: relative;
    overflow: hidden;
    padding: 1.15rem;
    border-radius: 24px;
    background: linear-gradient(180deg, rgba(10, 18, 27, 0.96), rgba(6, 11, 18, 0.94));
    border: 1px solid rgba(136, 221, 233, 0.12);
    box-shadow: var(--shadow);
}

.hero-meta-card::after,
.metric-card::after,
.glass-card::after {
    content: "";
    position: absolute;
    inset: 0;
    border-radius: inherit;
    background: linear-gradient(135deg, rgba(109, 247, 255, 0.08), transparent 25%, transparent 78%, rgba(109, 247, 255, 0.03));
    pointer-events: none;
}

.hero-meta-card span,
.metric-card span {
    display: block;
    font-size: 0.72rem;
    letter-spacing: 0.16em;
    text-transform: uppercase;
    color: #84a8b4;
}

.hero-meta-card strong,
.metric-card strong {
    display: block;
    margin-top: 0.45rem;
    font-family: "Space Grotesk", sans-serif;
    font-size: 1.55rem;
    letter-spacing: -0.04em;
    color: #f1fdff;
}

.section-header {
    display: flex;
    align-items: flex-end;
    justify-content: space-between;
    gap: 1rem;
    margin: 2.7rem 0 1.2rem;
}

.section-header-copy {
    max-width: 760px;
}

.section-kicker {
    display: inline-flex;
    align-items: center;
    gap: 0.5rem;
    color: var(--cyan-bright);
    font-size: 0.72rem;
    letter-spacing: 0.17em;
    text-transform: uppercase;
    margin-bottom: 0.35rem;
}

.section-title {
    margin: 0;
    font-family: "Space Grotesk", sans-serif;
    font-size: clamp(1.9rem, 4vw, 3rem);
    letter-spacing: -0.04em;
    color: #eafcff;
}

.section-copy {
    margin: 0.55rem 0 0;
    color: #8eaeb9;
    line-height: 1.7;
    font-size: 0.98rem;
}

.status-pill {
    display: inline-flex;
    align-items: center;
    gap: 0.45rem;
    padding: 0.45rem 0.75rem;
    border-radius: 999px;
    border: 1px solid rgba(77, 239, 255, 0.22);
    background: rgba(8, 20, 26, 0.82);
    color: var(--cyan-bright);
    text-transform: uppercase;
    letter-spacing: 0.15em;
    font-size: 0.69rem;
    white-space: nowrap;
}

.status-pill::before {
    content: "";
    width: 0.42rem;
    height: 0.42rem;
    border-radius: 999px;
    background: currentColor;
    box-shadow: 0 0 12px currentColor;
}

.metric-grid {
    display: grid;
    grid-template-columns: repeat(4, minmax(0, 1fr));
    gap: 0.9rem;
    margin-bottom: 1rem;
}

.metric-card p {
    margin: 0.5rem 0 0;
    color: #7e9ca7;
    line-height: 1.55;
    font-size: 0.9rem;
}

.media-card-header {
    margin-bottom: 0.85rem;
}

.card-eyebrow {
    display: inline-flex;
    color: var(--cyan-bright);
    font-size: 0.68rem;
    letter-spacing: 0.16em;
    text-transform: uppercase;
}

.media-card-title {
    margin: 0.2rem 0 0;
    font-family: "Space Grotesk", sans-serif;
    font-size: 1.35rem;
    letter-spacing: -0.03em;
    color: #e8fbff;
}

.media-card-subtitle {
    margin: 0.45rem 0 0;
    color: #81a0ab;
    line-height: 1.6;
    font-size: 0.92rem;
}

.media-frame,
.tensor-shot img {
    width: 100%;
    display: block;
    border-radius: 18px;
    border: 1px solid rgba(136, 221, 233, 0.14);
    background: #03070d;
}

.card-note {
    margin-top: 0.85rem;
    color: #9fc0c9;
    font-size: 0.9rem;
    line-height: 1.65;
}

.card-note strong {
    color: var(--cyan-bright);
}

.empty-workbench {
    position: relative;
    padding: 2rem 1.8rem 1.6rem;
    min-height: 22rem;
    background:
        linear-gradient(180deg, rgba(9, 16, 24, 0.95), rgba(7, 12, 19, 0.94)),
        linear-gradient(rgba(77, 239, 255, 0.04) 1px, transparent 1px),
        linear-gradient(90deg, rgba(77, 239, 255, 0.04) 1px, transparent 1px);
    background-size: auto, 32px 32px, 32px 32px;
}

.empty-workbench::before,
.empty-workbench::after {
    content: "";
    position: absolute;
    width: 20px;
    height: 20px;
    border-color: var(--cyan-bright);
    border-style: solid;
    border-width: 0;
}

.empty-workbench::before {
    inset: 16px auto auto 16px;
    border-top-width: 2px;
    border-left-width: 2px;
}

.empty-workbench::after {
    inset: auto 16px 16px auto;
    border-right-width: 2px;
    border-bottom-width: 2px;
}

.empty-center {
    max-width: 540px;
    margin: 2.6rem auto 0;
    text-align: center;
}

.empty-icon {
    width: 72px;
    height: 72px;
    margin: 0 auto 1rem;
    display: grid;
    place-items: center;
    border-radius: 20px;
    border: 1px solid rgba(136, 221, 233, 0.18);
    background: rgba(20, 34, 48, 0.65);
    color: var(--cyan-bright);
    font-size: 1.8rem;
    box-shadow: 0 12px 32px rgba(0, 0, 0, 0.28);
}

.empty-title {
    margin: 0;
    font-family: "Space Grotesk", sans-serif;
    font-size: clamp(1.8rem, 4vw, 2.7rem);
    letter-spacing: -0.04em;
    color: #dcfbff;
}

.empty-copy {
    margin: 0.8rem 0 0;
    color: #92b4bf;
    line-height: 1.75;
    font-size: 0.98rem;
}

.meta-strip {
    display: flex;
    justify-content: space-between;
    gap: 0.9rem;
    flex-wrap: wrap;
    margin-top: 2rem;
    color: #7a98a3;
    font-size: 0.74rem;
    text-transform: uppercase;
    letter-spacing: 0.14em;
}

.meta-strip span {
    white-space: nowrap;
}

.session-shell {
    margin-top: 1.4rem;
}

.session-badge {
    display: inline-flex;
    align-items: center;
    gap: 0.45rem;
    padding: 0.42rem 0.72rem;
    border-radius: 999px;
    border: 1px solid rgba(77, 239, 255, 0.16);
    background: rgba(7, 21, 29, 0.75);
    color: var(--cyan-bright);
    font-size: 0.68rem;
    letter-spacing: 0.16em;
    text-transform: uppercase;
}

.session-title {
    margin: 0.85rem 0 0;
    font-family: "Space Grotesk", sans-serif;
    font-size: clamp(2rem, 5vw, 3.3rem);
    letter-spacing: -0.05em;
    color: #ecfdff;
}

.session-copy {
    margin: 0.55rem 0 1.25rem;
    color: #95b5bf;
    max-width: 720px;
    line-height: 1.7;
}

.control-caption {
    margin-top: 0.9rem;
    color: #7998a2;
    font-size: 0.88rem;
}

.upload-label {
    margin: 0.4rem 0 0.7rem;
    color: #dffaff;
    font-size: 0.96rem;
    font-weight: 600;
    letter-spacing: 0.02em;
}

.tensor-header,
.result-header {
    display: flex;
    justify-content: space-between;
    gap: 1rem;
    align-items: flex-start;
    margin-bottom: 0.85rem;
}

.panel-tag {
    color: var(--cyan-bright);
    font-size: 0.68rem;
    letter-spacing: 0.16em;
    text-transform: uppercase;
    white-space: nowrap;
}

.panel-title {
    margin: 0;
    font-family: "Space Grotesk", sans-serif;
    font-size: 1.45rem;
    color: #e9fcff;
    letter-spacing: -0.03em;
}

.panel-subtitle {
    margin: 0.35rem 0 0;
    color: #84a5af;
    line-height: 1.55;
    font-size: 0.9rem;
}

.tensor-grid {
    display: grid;
    grid-template-columns: repeat(2, minmax(0, 1fr));
    gap: 0.95rem;
}

.tensor-shot span {
    display: block;
    margin-top: 0.55rem;
    color: #8ba9b3;
    font-size: 0.78rem;
    letter-spacing: 0.11em;
    text-transform: uppercase;
}

.result-value {
    margin-top: 0.55rem;
    font-family: "Space Grotesk", sans-serif;
    font-size: clamp(2.8rem, 6vw, 4.5rem);
    line-height: 0.95;
    letter-spacing: -0.06em;
    color: var(--cyan);
    text-shadow: 0 0 24px rgba(77, 239, 255, 0.18);
}

.result-summary {
    margin-top: 0.8rem;
    color: #9dc0c9;
    line-height: 1.65;
    font-size: 0.93rem;
}

.signal-stack {
    margin-top: 1.15rem;
}

.signal-row {
    display: flex;
    justify-content: space-between;
    align-items: center;
    gap: 0.6rem;
    margin: 0.65rem 0 0.28rem;
    color: #b7d8df;
    font-size: 0.8rem;
    letter-spacing: 0.12em;
    text-transform: uppercase;
}

.signal-track {
    width: 100%;
    height: 10px;
    border-radius: 999px;
    background: rgba(77, 239, 255, 0.08);
    overflow: hidden;
    border: 1px solid rgba(136, 221, 233, 0.08);
}

.signal-fill {
    height: 100%;
    border-radius: inherit;
    background: linear-gradient(90deg, #0bb9d9 0%, #71f6ff 100%);
}

.signal-fill.alt {
    background: linear-gradient(90deg, #274a55 0%, #7aa8b7 100%);
}

.tiny-ledger {
    margin-top: 1rem;
    color: #7f9ca7;
    font-size: 0.82rem;
    line-height: 1.65;
}

.tiny-ledger strong {
    color: #c5f6fb;
}

.methodology-card {
    margin-top: 1rem;
}

.methodology-grid {
    display: grid;
    grid-template-columns: repeat(3, minmax(0, 1fr));
    gap: 0.95rem;
    margin-top: 1rem;
}

.methodology-unit {
    padding: 0.9rem;
    border-radius: 18px;
    border: 1px solid rgba(136, 221, 233, 0.1);
    background: rgba(12, 20, 31, 0.74);
}

.methodology-unit span {
    display: block;
    color: var(--cyan-bright);
    text-transform: uppercase;
    letter-spacing: 0.15em;
    font-size: 0.68rem;
}

.methodology-unit p {
    margin: 0.45rem 0 0;
    color: #9ebdc6;
    font-size: 0.9rem;
    line-height: 1.65;
}

.footer-shell {
    margin-top: 3rem;
    padding: 1.35rem 1.5rem;
    border-radius: 24px;
    background: linear-gradient(180deg, rgba(9, 16, 24, 0.96), rgba(6, 10, 16, 0.94));
    border: 1px solid rgba(136, 221, 233, 0.12);
    display: flex;
    align-items: center;
    justify-content: space-between;
    gap: 1rem;
    flex-wrap: wrap;
}

.footer-brand {
    font-family: "Space Grotesk", sans-serif;
    font-size: 1.55rem;
    font-weight: 700;
    letter-spacing: -0.04em;
    color: #dcfbff;
}

.footer-links {
    display: flex;
    align-items: center;
    gap: 1rem;
    flex-wrap: wrap;
}

.footer-links a {
    text-decoration: none;
    color: #8daab5;
    font-size: 0.86rem;
}

.footer-copy {
    color: #73909a;
    font-size: 0.78rem;
    text-transform: uppercase;
    letter-spacing: 0.14em;
}

button[kind],
.stDownloadButton button {
    min-height: 3.1rem;
    border-radius: 16px !important;
    border: 1px solid rgba(136, 221, 233, 0.18) !important;
    background: linear-gradient(180deg, rgba(12, 21, 32, 0.96), rgba(8, 13, 21, 0.96)) !important;
    color: #e1fbff !important;
    text-transform: uppercase !important;
    letter-spacing: 0.16em !important;
    font-size: 0.74rem !important;
    font-weight: 700 !important;
    box-shadow: 0 16px 36px rgba(0, 0, 0, 0.22) !important;
}

button[kind]:hover,
.stDownloadButton button:hover {
    border-color: rgba(109, 247, 255, 0.34) !important;
    color: #ffffff !important;
}

div[data-testid="stDownloadButton"] > button {
    width: 100%;
}

[data-testid="stFileUploaderDropzone"] {
    border-radius: 18px !important;
    border: 1px dashed rgba(109, 247, 255, 0.25) !important;
    background: rgba(7, 16, 24, 0.8) !important;
}

[data-testid="stFileUploaderDropzoneInstructions"] span,
[data-testid="stFileUploaderDropzoneInstructions"] small,
[data-testid="stMarkdownContainer"] p {
    color: inherit;
}

[data-testid="stFileUploaderFileName"] {
    color: #d8fbff !important;
}

[data-testid="stFileUploader"] label,
[data-testid="stFileUploader"] label p,
[data-testid="stFileUploader"] label span,
[data-testid="stFileUploader"] small {
    color: #dffaff !important;
}

[data-testid="stFileUploader"] section small,
[data-testid="stFileUploader"] section span {
    color: #b8edf4 !important;
}

[data-testid="stExpander"] {
    border-radius: 24px !important;
    border: 1px solid rgba(136, 221, 233, 0.12) !important;
    background: linear-gradient(180deg, rgba(10, 18, 27, 0.96), rgba(6, 11, 18, 0.94)) !important;
    box-shadow: var(--shadow);
    overflow: hidden;
}

[data-testid="stExpander"] details {
    background: transparent !important;
}

[data-testid="stExpander"] summary {
    background: transparent !important;
    color: #e7fbff !important;
}

[data-testid="stExpander"] summary:hover {
    color: #ffffff !important;
}

[data-testid="stExpander"] summary p,
[data-testid="stExpander"] summary span,
[data-testid="stExpander"] label {
    color: #e7fbff !important;
}

[data-testid="stExpander"] details[open] summary {
    border-bottom: 1px solid rgba(136, 221, 233, 0.12);
}

[data-testid="stExpander"] svg {
    fill: #7ef7ff !important;
}

.stAlert {
    border-radius: 20px;
    background: rgba(9, 18, 28, 0.88);
    border: 1px solid rgba(136, 221, 233, 0.16);
}

.summary-table-wrap {
    margin-top: 0.6rem;
    padding: 0.9rem 1rem 1rem;
    border-radius: 16px;
    border: 1px solid rgba(136, 221, 233, 0.1);
    background: rgba(9, 14, 22, 0.92);
    overflow-x: auto;
    overflow-y: hidden;
}

.summary-table-title {
    margin: 0 0 0.8rem;
    color: #effdff;
    font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, "Liberation Mono", "Courier New", monospace;
    font-size: 0.98rem;
    letter-spacing: 0;
}

.summary-table {
    width: 100%;
    min-width: 760px;
    border-collapse: collapse;
    table-layout: fixed;
    border: 2px solid rgba(240, 253, 255, 0.88);
}

.summary-table thead th {
    padding: 0.72rem 0.85rem;
    text-align: left;
    color: #f5feff;
    font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, "Liberation Mono", "Courier New", monospace;
    font-size: 0.98rem;
    font-weight: 700;
    letter-spacing: 0;
    border-right: 2px solid rgba(240, 253, 255, 0.88);
    border-bottom: 2px solid rgba(240, 253, 255, 0.88);
    background: rgba(255, 255, 255, 0.02);
}

.summary-table thead th:last-child {
    border-right: none;
    text-align: right;
}

.summary-table tbody td {
    padding: 0.7rem 0.85rem;
    color: #eefbff;
    font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, "Liberation Mono", "Courier New", monospace;
    font-size: 0.94rem;
    line-height: 1.25;
    border-right: 2px solid rgba(240, 253, 255, 0.88);
    border-bottom: 2px solid rgba(240, 253, 255, 0.88);
    vertical-align: top;
    background: transparent;
}

.summary-table tbody td:last-child {
    border-right: none;
    text-align: right;
}

.summary-table tbody tr:last-child td {
    border-bottom: none;
}

.summary-layer-name,
.summary-shape,
.summary-param {
    font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, "Liberation Mono", "Courier New", monospace;
    color: #eefbff;
    font-size: 0.94rem;
}

.summary-layer-cell {
    white-space: normal;
}

.summary-totals {
    margin-top: 0.8rem;
    color: #eefbff;
    font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, "Liberation Mono", "Courier New", monospace;
    font-size: 0.94rem;
    line-height: 1.35;
}

@keyframes heroDrift {
    0% {
        transform: translateX(-2%) translateY(-1%) scale(1);
    }
    50% {
        transform: translateX(2%) translateY(1%) scale(1.02);
    }
    100% {
        transform: translateX(-2%) translateY(-1%) scale(1);
    }
}

@media (max-width: 980px) {
    .top-nav,
    .section-header,
    .footer-shell {
        flex-direction: column;
        align-items: flex-start;
    }

    .hero-meta,
    .metric-grid,
    .methodology-grid,
    .tensor-grid {
        grid-template-columns: 1fr;
    }

    .nav-links {
        justify-content: flex-start;
    }
}
</style>
"""


@st.cache_resource
def load_model() -> tf.keras.Model:
    if not FINAL_MODEL_PATH.exists():
        raise FileNotFoundError(
            "No se encontró el modelo entrenado en models/model.keras. Ejecute scripts/train_pipeline.py."
        )
    return tf.keras.models.load_model(FINAL_MODEL_PATH, compile=False)


@st.cache_data
def load_results_metadata() -> dict[str, Any]:
    if RESULTS_JSON_PATH.exists():
        return json.loads(RESULTS_JSON_PATH.read_text(encoding="utf-8"))
    return {}


@st.cache_data
def load_model_summary_text() -> str:
    if MODEL_SUMMARY_PATH.exists():
        return MODEL_SUMMARY_PATH.read_text(encoding="utf-8")
    return ""


def inject_css() -> None:
    st.markdown(CUSTOM_CSS, unsafe_allow_html=True)


def bootstrap_state() -> None:
    if "active_image_bytes" not in st.session_state:
        st.session_state.active_image_bytes = None
    if "active_image_name" not in st.session_state:
        st.session_state.active_image_name = None
    if "uploader_nonce" not in st.session_state:
        st.session_state.uploader_nonce = 0


def to_data_url(image_source: Image.Image | np.ndarray | Path) -> str:
    if isinstance(image_source, Path):
        raw_bytes = image_source.read_bytes()
        suffix = image_source.suffix.lower().replace(".", "") or "png"
        encoded = base64.b64encode(raw_bytes).decode("utf-8")
        return f"data:image/{suffix};base64,{encoded}"

    if isinstance(image_source, np.ndarray):
        image = Image.fromarray(image_source.astype(np.uint8))
    else:
        image = image_source

    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return f"data:image/png;base64,{encoded}"


def format_pct(value: float | None) -> str:
    if value is None:
        return "--"
    return f"{value:.2%}"


def label_display(label: str) -> str:
    return {"female": "FEMENINO", "male": "MASCULINO"}.get(label, label.upper().replace("_", " "))


def experiment_display_name(name: str) -> str:
    return {"regularized": "REGULARIZADO", "baseline": "BASE"}.get(name, name.upper())


def parse_model_summary(summary_text: str) -> dict[str, Any]:
    parsed: dict[str, Any] = {
        "model_name": None,
        "layers": [],
        "totals": {},
    }
    if not summary_text.strip():
        return parsed

    current_layer: dict[str, str] | None = None
    for raw_line in summary_text.splitlines():
        line = raw_line.rstrip()
        stripped = line.strip()
        if not stripped:
            continue

        model_match = re.match(r'^Model:\s*"([^"]+)"$', stripped)
        if model_match:
            parsed["model_name"] = model_match.group(1)
            continue

        total_match = re.match(
            r"^(Total params|Trainable params|Non-trainable params|Optimizer params):\s*(.+)$",
            stripped,
        )
        if total_match:
            parsed["totals"][total_match.group(1)] = total_match.group(2).strip()
            continue

        if "│" not in line and "┃" not in line:
            continue

        cells = [cell.strip() for cell in re.split(r"[│┃]", line)[1:-1]]
        if not cells or cells[0].startswith("Layer"):
            continue

        first = cells[0].strip()
        second = cells[1].strip() if len(cells) > 1 else ""
        third = cells[2].strip() if len(cells) > 2 else ""

        if first.startswith("(") and first.endswith(")") and current_layer is not None:
            current_layer["tipo"] = first[1:-1].strip()
            continue

        layer_name = first
        layer_type = ""
        layer_match = re.match(r"(.+?)\s*\(([^()]*)\)$", first)
        if layer_match:
            layer_name = layer_match.group(1).strip()
            layer_type = layer_match.group(2).strip()

        current_layer = {
            "capa": layer_name or "--",
            "tipo": layer_type or "--",
            "salida": second or "--",
            "parametros": third or "--",
        }
        parsed["layers"].append(current_layer)

    return parsed


def extract_total_params(summary_text: str) -> str:
    match = re.search(r"Total params:\s*([\d,]+)", summary_text)
    return match.group(1) if match else "--"


def saliency_summary(heatmap: np.ndarray) -> str:
    _ = heatmap
    return (
        "Las zonas con colores mas calientes indican los pixeles ante los que la salida del modelo es mas sensible. "
        "Si aparecen en amarillo, naranja o rojo, significa que pequenos cambios en esa region modifican mas la prediccion."
    )


def gradcam_summary(heatmap: np.ndarray, confidence: float) -> str:
    _ = heatmap
    _ = confidence
    return (
        "Las zonas mas calientes muestran donde las capas profundas activan con mayor fuerza para sostener la clase predicha. "
        "Las regiones frias tienen menor aporte en la decision final del modelo."
    )


def set_active_upload(uploaded_file) -> None:
    st.session_state.active_image_bytes = uploaded_file.getvalue()
    st.session_state.active_image_name = uploaded_file.name


def clear_active_image() -> None:
    st.session_state.active_image_bytes = None
    st.session_state.active_image_name = None
    st.session_state.uploader_nonce += 1


def get_active_image() -> tuple[Image.Image | None, str | None]:
    if not st.session_state.active_image_bytes:
        return None, None
    image = Image.open(io.BytesIO(st.session_state.active_image_bytes))
    image.load()
    return image, st.session_state.active_image_name


def analyze_image(pil_image: Image.Image) -> dict[str, Any]:
    model = load_model()
    original_rgb, resized_rgb, input_tensor = prepare_image_from_pil(pil_image, DEFAULT_IMAGE_SIZE)
    prob_male = float(model.predict(input_tensor, verbose=0)[0][0])
    prob_female = 1.0 - prob_male
    predicted_label = CLASS_NAMES[int(prob_male >= 0.5)]
    alternative_label = "female" if predicted_label == "male" else "male"
    confidence = max(prob_male, prob_female)
    saliency = make_saliency_map(model, input_tensor)
    gradcam = make_gradcam_heatmap(model, input_tensor, last_conv_layer_name="last_conv")
    return {
        "original_rgb": original_rgb,
        "resized_rgb": resized_rgb,
        "prob_male": prob_male,
        "prob_female": prob_female,
        "predicted_label": predicted_label,
        "alternative_label": alternative_label,
        "confidence": confidence,
        "saliency": saliency,
        "gradcam": gradcam,
        "saliency_overlay": overlay_heatmap(original_rgb, saliency),
        "gradcam_overlay": overlay_heatmap(original_rgb, gradcam),
    }


def render_section_header(kicker: str, title: str, copy: str, status: str | None = None) -> None:
    status_html = f'<div class="status-pill">{status}</div>' if status else ""
    st.markdown(
        f"""
        <div class="section-header">
            <div class="section-header-copy">
                <div class="section-kicker">{kicker}</div>
                <h2 class="section-title">{title}</h2>
                <p class="section-copy">{copy}</p>
            </div>
            {status_html}
        </div>
        """,
        unsafe_allow_html=True,
    )


def render_navbar() -> None:
    st.markdown(
        """
        <div id="top" class="section-anchor"></div>
        <nav class="top-nav">
            <a class="brand-lockup" href="#top">PRISMA XAI LAB</a>
            <div class="nav-links">
                <a href="#modelo">Modelo</a>
                <a href="#datos">Datos</a>
                <a href="#analisis">Analisis</a>
                <a href="#reportes">Reportes</a>
            </div>
        </nav>
        """,
        unsafe_allow_html=True,
    )


def render_hero(metadata: dict[str, Any]) -> None:
    accuracy = format_pct(metadata.get("test_metrics", {}).get("accuracy"))
    auc = metadata.get("test_metrics", {}).get("auc")
    auc_text = f"{auc:.3f}" if auc is not None else "--"
    best_experiment = experiment_display_name(metadata.get("best_experiment", {}).get("name", "regularized"))
    subset_count = metadata.get("modeling_dataset_count", "--")

    st.markdown(
        f"""
        <section class="hero-shell">
            <div class="hero-content">
                <div class="hero-badge">Vision explicable en linea</div>
                <h1 class="hero-title">ARQUITECTURA VISUAL <span>DEL SEXO</span></h1>
                <p class="hero-copy">
                    Una CNN entrenada desde cero se combina con explicabilidad visual para clasificar
                    rostros en las clases <strong>femenino</strong> y <strong>masculino</strong> con evidencia
                    interpretable.
                </p>
                <div class="hero-actions">
                    <a class="hero-button primary" href="#analisis">Ir al analisis</a>
                    <a class="hero-button secondary" href="#modelo">Ver modelo</a>
                </div>
            </div>
            <div class="hero-meta">
                <div class="hero-meta-card">
                    <span>Exactitud de prueba</span>
                    <strong>{accuracy}</strong>
                </div>
                <div class="hero-meta-card">
                    <span>AUC de prueba</span>
                    <strong>{auc_text}</strong>
                </div>
                <div class="hero-meta-card">
                    <span>Mejor experimento</span>
                    <strong>{best_experiment}</strong>
                </div>
                <div class="hero-meta-card">
                    <span>Submuestra de modelado</span>
                    <strong>{subset_count}</strong>
                </div>
            </div>
        </section>
        """,
        unsafe_allow_html=True,
    )


def metric_card_html(label: str, value: str, note: str) -> str:
    return f"""
    <div class="metric-card">
        <span>{label}</span>
        <strong>{value}</strong>
        <p>{note}</p>
    </div>
    """


def render_metric_row(cards: list[tuple[str, str, str]]) -> None:
    columns = st.columns(len(cards), gap="medium")
    for column, (label, value, note) in zip(columns, cards):
        with column:
            st.markdown(metric_card_html(label, value, note), unsafe_allow_html=True)


def render_media_card(
    title: str,
    subtitle: str,
    image_source: Path | np.ndarray | Image.Image,
    note: str = "",
    eyebrow: str = "",
) -> None:
    note_html = f'<div class="card-note">{note}</div>' if note else ""
    eyebrow_html = f'<div class="card-eyebrow">{eyebrow}</div>' if eyebrow else ""
    st.markdown(
        f"""
        <div class="glass-card">
            <div class="media-card-header">
                {eyebrow_html}
                <h3 class="media-card-title">{title}</h3>
                <p class="media-card-subtitle">{subtitle}</p>
            </div>
            <img class="media-frame" src="{to_data_url(image_source)}" alt="{title}" />
            {note_html}
        </div>
        """,
        unsafe_allow_html=True,
    )


def render_model_summary_table(summary_text: str) -> None:
    parsed = parse_model_summary(summary_text)
    if not parsed["layers"]:
        st.info("No hay un resumen estructurado disponible del modelo.")
        return

    model_name = html.escape(parsed["model_name"] or "cnn_gender_classifier")
    rows_html = "".join(
        f"""
        <tr>
            <td class="summary-layer-cell">
                <span class="summary-layer-name">{html.escape(layer['capa'] + (f" ({layer['tipo']})" if layer['tipo'] and layer['tipo'] != '--' else ''))}</span>
            </td>
            <td class="summary-shape">{html.escape(layer['salida'])}</td>
            <td class="summary-param">{html.escape(layer['parametros'])}</td>
        </tr>
        """
        for layer in parsed["layers"]
    )

    total_lines = [
        ("Total params", "Total params"),
        ("Trainable params", "Trainable params"),
        ("Non-trainable params", "Non-trainable params"),
        ("Optimizer params", "Optimizer params"),
    ]
    totals_html = "<br/>".join(
        f"{label_es}: {html.escape(parsed['totals'].get(label_en, '--'))}"
        for label_en, label_es in total_lines
        if parsed["totals"].get(label_en)
    )

    st.markdown(
        f"""
        <div class="summary-table-wrap">
            <div class="summary-table-title">Model: "{model_name}"</div>
            <table class="summary-table">
                <thead>
                    <tr>
                        <th style="width: 47%;">Layer (type)</th>
                        <th style="width: 33%;">Output Shape</th>
                        <th style="width: 20%;">Param #</th>
                    </tr>
                </thead>
                <tbody>
                    {rows_html}
                </tbody>
            </table>
            <div class="summary-totals">
                {totals_html}
            </div>
        </div>
        """,
        unsafe_allow_html=True,
    )


def render_models_section(metadata: dict[str, Any], total_params: str, summary_text: str) -> None:
    st.markdown('<div id="modelo" class="section-anchor"></div>', unsafe_allow_html=True)
    render_section_header(
        "Ficha del modelo",
        "Arquitectura experimental y evidencia de entrenamiento",
        "El clasificador se entreno desde cero en TensorFlow-Keras, se comparo en dos configuraciones y la variante regularizada fue seleccionada como artefacto final para el despliegue.",
    )

    best_experiment = metadata.get("best_experiment", {})
    test_metrics = metadata.get("test_metrics", {})
    render_metric_row(
        [
            ("Tensor de entrada", f"{DEFAULT_IMAGE_SIZE[0]}x{DEFAULT_IMAGE_SIZE[1]}", "Imagen RGB con resize_with_pad y normalizacion en coma flotante."),
            ("Parametros totales", total_params, "Arquitectura compacta pensada para mantenerse desplegable en Streamlit Cloud."),
            ("Mejor tasa de aprendizaje", str(best_experiment.get("learning_rate", "--")), "Valor elegido en el experimento regularizado ganador."),
            ("Perdida de prueba", f"{test_metrics.get('loss', 0):.4f}" if test_metrics.get("loss") is not None else "--", "Perdida final sobre el conjunto de prueba reservado."),
        ]
    )

    col_left, col_right = st.columns(2, gap="large")
    with col_left:
        if FIGURE_PATHS["hyperparameter_comparison"].exists():
            render_media_card(
                "Comparacion de hiperparametros",
                "Curvas de exactitud y perdida de validacion para la configuracion base y la regularizada.",
                FIGURE_PATHS["hyperparameter_comparison"],
                note=(
                    f"<strong>Conclusion:</strong> La configuracion <strong>{experiment_display_name(best_experiment.get('name', 'regularized'))}</strong> "
                    f"fue la ganadora con una mejor exactitud de validacion de {format_pct(best_experiment.get('best_val_accuracy'))}."
                ),
                eyebrow="Exploracion experimental",
            )
    with col_right:
        if FIGURE_PATHS["training_final"].exists():
            render_media_card(
                "Trayectoria final de entrenamiento",
                "Curvas de aprendizaje del experimento seleccionado para despliegue.",
                FIGURE_PATHS["training_final"],
                note=(
                    f"<strong>Punto de control:</strong> mejor epoca {best_experiment.get('best_epoch', '--')} con "
                    f"perdida de validacion de {best_experiment.get('best_val_loss', 0):.4f}."
                    if best_experiment.get("best_val_loss") is not None
                    else ""
                ),
                eyebrow="Curva de entrenamiento",
            )

    with st.expander("Resumen textual de la arquitectura"):
        render_model_summary_table(summary_text)


def render_datasets_section(metadata: dict[str, Any]) -> None:
    st.markdown('<div id="datos" class="section-anchor"></div>', unsafe_allow_html=True)
    render_section_header(
        "Inteligencia de datos",
        "Cobertura, estratificacion y logica de muestreo",
        "La etapa exploratoria reviso el conjunto completo de rostros femenino/masculino, mientras que el modelado utilizo una submuestra balanceada y estratificada para mantener el entrenamiento eficiente y reproducible.",
    )

    dataset_summary = metadata.get("dataset_summary", {})
    female_count = dataset_summary.get("female", {}).get("count", "--")
    male_count = dataset_summary.get("male", {}).get("count", "--")
    split_counts = metadata.get("split_counts", {})
    split_text = f"{split_counts.get('train', '--')} / {split_counts.get('validation', '--')} / {split_counts.get('test', '--')}"

    render_metric_row(
        [
            ("Imagenes femeninas", f"{female_count}", "Conteo observado durante la inspeccion del conjunto original."),
            ("Imagenes masculinas", f"{male_count}", "Conteo observado durante la inspeccion del conjunto original."),
            ("Submuestra de modelado", f"{metadata.get('modeling_dataset_count', '--')}", "Submuestra balanceada usada para entrenamiento, validacion y prueba."),
            ("Particiones T/V/P", split_text, "Cantidad de ejemplos en entrenamiento, validacion y prueba."),
        ]
    )

    col_left, col_right = st.columns(2, gap="large")
    with col_left:
        if FIGURE_PATHS["dataset_mosaic"].exists():
            render_media_card(
                "Mosaico del conjunto",
                "Vista cualitativa de la variabilidad observada durante la etapa exploratoria.",
                FIGURE_PATHS["dataset_mosaic"],
                note="El mosaico ayuda a revisar poses, iluminacion y textura antes de iniciar el ajuste del modelo.",
                eyebrow="Lectura cualitativa",
            )
    with col_right:
        if FIGURE_PATHS["class_distribution"].exists():
            render_media_card(
                "Distribucion de clases",
                "Conteos observados por etiqueta antes de construir la submuestra.",
                FIGURE_PATHS["class_distribution"],
                note="La submuestra de modelado se obligo a permanecer balanceada aunque el conjunto fuente solo este aproximadamente balanceado.",
                eyebrow="Auditoria de balance",
            )


def render_empty_workbench() -> None:
    st.markdown(
        """
        <div class="glass-card empty-workbench">
            <div class="empty-center">
                <div class="empty-icon">⌘</div>
                <h3 class="empty-title">Sube La Imagen Para Analizar</h3>
                <p class="empty-copy">
                    La aplicacion recibe imagenes faciales en formato JPG, JPEG o PNG, las convierte
                    al tensor 96x96 RGB del modelo y ejecuta automaticamente la prediccion junto con
                    Saliency Map y Grad-CAM.
                </p>
            </div>
            <div class="meta-strip">
                <span>Preprocesamiento: RGB + padding</span>
                <span>Visualizacion: Saliency + Grad-CAM</span>
                <span>Modelo: CNN + XAI v1.0</span>
            </div>
        </div>
        """,
        unsafe_allow_html=True,
    )


def render_analysis_header(source_name: str) -> None:
    st.markdown(
        f"""
        <div class="session-shell">
            <div class="session-badge">Analisis activo</div>
            <h3 class="session-title">Resultado de la inferencia</h3>
            <p class="session-copy">
                Se esta evaluando la activacion de la red sobre la imagen <strong>{source_name}</strong>
                para producir una decision de clasificacion por sexo y su evidencia visual asociada.
            </p>
        </div>
        """,
        unsafe_allow_html=True,
    )


def render_input_tensor_card(original_rgb: np.ndarray, resized_rgb: np.ndarray, source_name: str) -> None:
    st.markdown(
        f"""
        <div class="glass-card">
            <div class="tensor-header">
                <div>
                    <h3 class="panel-title">Analisis del tensor de entrada</h3>
                    <p class="panel-subtitle">Archivo fuente: {source_name}</p>
                </div>
                <div class="panel-tag">Fase I</div>
            </div>
            <div class="tensor-grid">
                <div class="tensor-shot">
                    <img src="{to_data_url(original_rgb)}" alt="Original input" />
                    <span>Imagen original</span>
                </div>
                <div class="tensor-shot">
                    <img src="{to_data_url(resized_rgb)}" alt="Preprocessed tensor" />
                    <span>Tensor 96x96</span>
                </div>
            </div>
        </div>
        """,
        unsafe_allow_html=True,
    )


def render_prediction_card(analysis: dict[str, Any], source_name: str) -> None:
    prob_female = analysis["prob_female"]
    prob_male = analysis["prob_male"]
    predicted = analysis["predicted_label"]
    alternative = analysis["alternative_label"]
    confidence = analysis["confidence"]
    margin = abs(prob_male - prob_female)
    confidence_note = (
        "Clasificacion de alta confianza"
        if confidence >= 0.75
        else "Clasificacion de confianza media"
        if confidence >= 0.60
        else "Clasificacion con margen estrecho"
    )
    predicted_probability = prob_male if predicted == "male" else prob_female
    alternative_probability = prob_female if predicted == "male" else prob_male

    st.markdown(
        f"""
        <div class="glass-card">
            <div class="result-header">
                <div>
                    <div class="card-eyebrow">Resultado de clasificacion</div>
                    <div class="result-value">{label_display(predicted)}</div>
                    <p class="panel-subtitle">{confidence_note} obtenido a partir de <strong>{source_name}</strong>.</p>
                </div>
                <div class="panel-tag">Inferencia en vivo</div>
            </div>
            <div class="signal-stack">
                <div class="signal-row">
                    <span>Confianza</span>
                    <span>{predicted_probability:.1%}</span>
                </div>
                <div class="signal-track">
                    <div class="signal-fill" style="width: {predicted_probability * 100:.2f}%"></div>
                </div>
                <div class="signal-row">
                    <span>Alternativa ({label_display(alternative)})</span>
                    <span>{alternative_probability:.1%}</span>
                </div>
                <div class="signal-track">
                    <div class="signal-fill alt" style="width: {alternative_probability * 100:.2f}%"></div>
                </div>
            </div>
            <div class="result-summary">
                La salida sigmoide del modelo esta calibrada sobre la categoria interna correspondiente a <strong>masculino</strong>.
                El margen de decision en este caso es de <strong>{margin:.1%}</strong>, por eso la probabilidad
                complementaria de la otra clase tambien se muestra para la interpretacion.
            </div>
            <div class="tiny-ledger">
                <strong>Femenino:</strong> {prob_female:.2%}<br/>
                <strong>Masculino:</strong> {prob_male:.2%}
            </div>
        </div>
        """,
        unsafe_allow_html=True,
    )


def render_methodology_card(metadata: dict[str, Any], analysis: dict[str, Any]) -> None:
    best = metadata.get("best_experiment", {})
    test_metrics = metadata.get("test_metrics", {})
    auc = test_metrics.get("auc")
    auc_text = f"{auc:.3f}" if auc is not None else "--"
    st.markdown(
        f"""
        <div class="glass-card methodology-card">
            <div class="media-card-header">
                <div class="card-eyebrow">Notas tecnicas y metodologia</div>
                <h3 class="media-card-title">Como se produjo esta inferencia</h3>
                <p class="media-card-subtitle">
                    Cada paso de abajo esta alineado con el pipeline de entrenamiento usado en el notebook y en la entrega final del laboratorio.
                </p>
            </div>
            <div class="methodology-grid">
                <div class="methodology-unit">
                    <span>Preprocesamiento</span>
                    <p>Conversion a RGB, normalizacion a float32 y <code>resize_with_pad</code> a {DEFAULT_IMAGE_SIZE[0]}x{DEFAULT_IMAGE_SIZE[1]} antes de entrar a la CNN.</p>
                </div>
                <div class="methodology-unit">
                    <span>Modelo</span>
                    <p>Mejor experimento: <strong>{experiment_display_name(best.get('name', 'regularized'))}</strong> con dropout {best.get('dropout_rate', '--')} y tasa de aprendizaje {best.get('learning_rate', '--')}.</p>
                </div>
                <div class="methodology-unit">
                    <span>Referencia</span>
                    <p>Exactitud de prueba de {format_pct(test_metrics.get('accuracy'))} y AUC de {auc_text} para el modelo desplegado.</p>
                </div>
            </div>
        </div>
        """,
        unsafe_allow_html=True,
    )


def render_workbench_section(metadata: dict[str, Any]) -> None:
    st.markdown('<div id="analisis" class="section-anchor"></div>', unsafe_allow_html=True)
    workbench_status = "Esperando imagen" if st.session_state.active_image_bytes is None else "Analisis cargado"
    render_section_header(
        "Flujo de inferencia",
        "Area de analisis",
        "Aqui ocurre el flujo central de la aplicacion: subir una imagen, preprocesarla, clasificarla y visualizar la evidencia de Saliency Map y Grad-CAM sobre la misma entrada analizada.",
        status=workbench_status,
    )

    uploader_key = f"main_uploader_{st.session_state.uploader_nonce}"
    st.markdown('<div class="upload-label">Selecciona o arrastra una imagen facial</div>', unsafe_allow_html=True)
    uploaded_file = st.file_uploader(
        "Cargador de imagen facial",
        type=["jpg", "jpeg", "png"],
        key=uploader_key,
        label_visibility="collapsed",
    )
    st.caption("El analisis comienza automaticamente cuando la imagen se carga.")

    if uploaded_file is not None:
        set_active_upload(uploaded_file)

    if st.session_state.active_image_bytes is None:
        render_empty_workbench()
        return

    render_analysis_header(st.session_state.active_image_name or "imagen cargada")
    source_cols = st.columns([1.45, 0.55], gap="medium")
    with source_cols[0]:
        st.markdown(
            '<p class="control-caption">Si subes otra imagen en el cargador de arriba, el analisis actual se reemplaza automaticamente por el nuevo.</p>',
            unsafe_allow_html=True,
        )
    with source_cols[1]:
        if st.button("Limpiar analisis", use_container_width=True):
            clear_active_image()
            st.rerun()

    active_image, source_name = get_active_image()
    if active_image is None or source_name is None:
        st.error("No fue posible recuperar la imagen activa de la sesion.")
        return

    with st.spinner("Ejecutando la inferencia y generando los mapas de atribucion..."):
        analysis = analyze_image(active_image)

    top_left, top_right = st.columns([1.35, 1.0], gap="large")
    with top_left:
        render_input_tensor_card(analysis["original_rgb"], analysis["resized_rgb"], source_name)
    with top_right:
        render_prediction_card(analysis, source_name)

    bottom_left, bottom_right = st.columns(2, gap="large")
    with bottom_left:
        render_media_card(
            "Saliency Map",
            "Gradiente de atribucion a nivel de pixel sobre el tensor exacto que consumio la CNN.",
            analysis["saliency_overlay"],
            note=f"<strong>Lectura:</strong> {saliency_summary(analysis['saliency'])}",
            eyebrow="Campo de atribucion",
        )
    with bottom_right:
        render_media_card(
            "Grad-CAM",
            "Mapa de activacion por clase construido desde las representaciones convolucionales profundas.",
            analysis["gradcam_overlay"],
            note=f"<strong>Lectura:</strong> {gradcam_summary(analysis['gradcam'], analysis['confidence'])}",
            eyebrow="Traza de activacion",
        )

    render_methodology_card(metadata, analysis)


def render_archives_section() -> None:
    st.markdown('<div id="reportes" class="section-anchor"></div>', unsafe_allow_html=True)
    render_section_header(
        "Artefactos y reporte",
        "Archivos del despliegue y evidencia visual",
        "Esta seccion expone figuras, notebook y metricas que respaldan la entrega final del laboratorio y sirven como soporte tecnico del despliegue.",
    )

    col_left, col_right = st.columns(2, gap="large")
    with col_left:
        if FIGURE_PATHS["confusion_matrix"].exists():
            render_media_card(
                "Matriz de confusion",
                "Desempeno sobre el conjunto de prueba por etiquetas reales y predichas.",
                FIGURE_PATHS["confusion_matrix"],
                note="Sirve para revisar si el clasificador tiende a favorecer una clase por encima de la otra.",
                eyebrow="Artefacto de evaluacion",
            )
    with col_right:
        if FIGURE_PATHS["xai_example"].exists():
            render_media_card(
                "Ejemplo de XAI de referencia",
                "Figura compuesta generada durante el pipeline del laboratorio para una imagen correctamente clasificada.",
                FIGURE_PATHS["xai_example"],
                note="Este artefacto muestra como luce una salida completa del pipeline de interpretabilidad fuera de la interfaz interactiva.",
                eyebrow="Artefacto de interpretabilidad",
            )

def render_footer() -> None:
    st.markdown(
        """
        <section class="footer-shell">
            <div class="footer-brand">PRISMA XAI LAB</div>
            <div class="footer-copy">2026 Prisma XAI Lab · clasificacion por sexo con CNN y explicabilidad visual</div>
        </section>
        """,
        unsafe_allow_html=True,
    )


def main() -> None:
    inject_css()
    bootstrap_state()
    metadata = load_results_metadata()
    summary_text = load_model_summary_text()
    total_params = extract_total_params(summary_text)

    render_navbar()
    render_hero(metadata)
    render_models_section(metadata, total_params, summary_text)
    render_datasets_section(metadata)

    try:
        render_workbench_section(metadata)
    except FileNotFoundError as error:
        st.error(str(error))

    render_archives_section()
    render_footer()


if __name__ == "__main__":
    main()
